# LRU的实现方式

- 基于有序字典的实现
- hash表 + 双向队列

## 有序字典

```
from collections import OrderedDict
```
**popitem(last=True)** 
- last=True  按照 LIFO 的顺序删除
- last=False 按照 FIFO 的顺序删除

**move_to_end(key, last=True)**
- last=True  按照 LIFO 的顺序将key对移动到末尾
- last=False 按照 FIFO 的顺序将key对移动到末尾


In [1]:
from collections import OrderedDict
class LRUCache(OrderedDict):

    def __init__(self, capacity):
        """
        :type capacity: int
        """
        self.capacity = capacity

    def get(self, key):
        """
        :type key: int
        :rtype: int
        """
        if key not in self:
            return - 1
        
        self.move_to_end(key)
        return self[key]

    def put(self, key, value):
        """
        :type key: int
        :type value: int
        :rtype: void
        """
        if key in self:
            self.move_to_end(key)
        self[key] = value
        if len(self) > self.capacity:
            self.popitem(last = False)

## 双向队列和hash表

因为同时使用了双向队列和hash表来维护LRU,
所以分别需要两套对应的删除和添加机制
- 双向链表的添加和删除
- hash表的添加和删除

LRU中添加操作和删除操作仅仅在put方法中出现

> Tips:  
> 使用等号进行交换时  
> 引用类型最好不要直接交换  
> 数值问题随意交换

### 如何初始化一个双向链表?

初始化头尾节点即可:
- 创建 head, tail
- 将头尾相连
- 完善双向链表的节点增加(头增加),节点删除(引用删除)

### 如何将双向链表转变为双向队列?

在节点增加节点删除的基础之上实现头部增加,尾巴移除

In [3]:
class DLinkNode:
    
    def __init__(self):
        self.key = 0
        self.value = 0
        self.prev = None
        self.next = None

class LRUCache:
    
    def _add_node(self,node):
        tmp = self.head.next
        
        self.head.next = node
        node.next = tmp
        tmp.prev = node
        node.prev = self.head
        
    def _remove_node(self,node):
        prev = node.prev
        next = node.next
        prev.next = next
        next.prev = prev
    
    def move_to_head(self,node):
        self._remove_node(node)
        self._add_node(node)
    
    def pop_item(self):
        node = self.tail.prev
        self._remove_node(node)
        return node
    
    def __init__(self,capacity):
        
        self.capacity = capacity
        self.size = 0
        self.cache = {}
        # 如何初始化一个双向链表
        self.head,self.tail = DLinkNode(),DLinkNode()
        self.head.next = self.tail
        self.tail.prev = self.head 
    
    def get(self,key):
        if key in self.cache:
            node = self.cache[key]
            self.move_to_head(node)
            return node.value
        else:
            return -1
            
    def put(self,key,value):
        
        if key in self.cache:
            node = self.cache[key]
            node.value = value
            self.move_to_head(node)
        else:
            new_node = DLinkNode()
            new_node.key,new_node.value = key,value
            self.size += 1
            self.cache[key] = new_node
            self._add_node(new_node)
            if self.size >= self.capacity:
                # 需要删除
                del_node = self.pop_item()
                del self.cache[del_node.key]
                self.size -= 1
                
    def show_me(self):
        cur = self.head
        ans = []
        while cur!=None:
            ans.append(str(cur.value))
            cur = cur.next
        print("".join(ans))

In [4]:
cache = LRUCache(2)
cache.put(1,1)
cache.put(2, 2)
cache.get(1)
cache.put(3, 3)
cache.get(2)

-1